# ĐỒ ÁN 2: DATA FITTING VÀ PHƯƠNG PHÁP OLS
### PHẦN 2: ỨNG DỤNG DATA FITTING VÀO DỮ LIỆU THỰC TẾ

* **Môn học:** Toán Ứng Dụng và Thống Kê
* **Mã môn:** MTH00051
* **Học kỳ:** Học kỳ 2, Năm học 2025 - 2026
* **Giảng viên lý thuyết:** ThS. Võ Nam Thục Đoan, ThS. Lê Nhựt Nam
* **Nhóm thực hiện:** Nhóm 14

---

**Mục tiêu:**
* Minh họa quy trình tiền xử lý và phân tích một bộ dữ liệu môi trường thực tế;
* Xây dựng và đánh giá các mô hình hồi quy tuyến tính từ đầu (OLS, Ridge, Lasso);
* Tạo tài liệu tham khảo để hiểu rõ cách áp dụng các kỹ thuật `data fitting` nhằm giải quyết các bài toán thực tiễn.



## 1. Mô tả bộ dữ liệu và Import thư viện

**Bài toán hướng đến:** Trong thực tế, các máy phân tích khí độc hóa học chuẩn tham chiếu thường rất đắt tiền và khó triển khai trên diện rộng, trong khi các cảm biến oxit kim loại tự động hóa lại rẻ tiền và dễ lắp đặt nhưng độ chính xác thô thấp hơn. Nhóm chúng em hướng đến bài toán: **Xây dựng mô hình toán học sử dụng phản hồi từ các cảm biến hóa học đo được trong không khí (PT08.S1 - PT08.S5) cùng các yếu tố khí hậu (Nhiệt độ, Độ ẩm) để dự đoán chính xác nồng độ của khí độc hại Carbon Monoxide (CO) `CO(GT)`**.

Đây là một ví dụ điển hình của bài toán `data fitting`, mục tiêu là tìm ra một hàm số hồi quy đa biến phù hợp nhất để mô tả mối quan hệ giữa biến đầu ra ($y$: Nồng độ khí Carbon Monoxide (CO) thực tế) và các biến đầu vào ($X$: Các đặc trưng từ hệ thống cảm biến).

**Quy ước:** Mức ý nghĩa áp dụng cho các kiểm định thống kê trong đồ án: $\alpha = 0.05$

*Lưu ý về dữ liệu:* Trong bộ dữ liệu gốc từ UCI, các giá trị bị thiếu (missing values) được hệ thống tự động mã hóa bằng giá trị `-200`. Do đó, trước khi tiến hành EDA, ta cần chuyển đổi toàn bộ các giá trị `-200` này về định dạng `NaN` chuẩn để không làm lệch các thống kê mô tả.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập seed cố định cho toàn bộ thực nghiệm Phần 2
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Cấu hình đồ họa hiển thị
plt.rcParams['figure.dpi'] = 100
sns.set_theme(style="whitegrid")

# LOAD DATASET VÀ XỬ LÝ -200
df = pd.read_csv("data/AirQualityUCI.csv")

# 1. Dọn dẹp các hàng/cột trống hoàn toàn do lỗi dấu phẩy thừa
df.dropna(how='all', axis=1, inplace=True)
df.dropna(how='all', axis=0, inplace=True)

# 2. Ép kiểu dữ liệu các cột về dạng số thực (trừ Date và Time)
for col in df.columns:
    if col not in ['Date', 'Time']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 3. Chuyển đổi mã lỗi -200 thành NaN (Missing value chuẩn của Pandas)
df.replace(-200, np.nan, inplace=True)



n_samples = df.shape[0]
p_features = df.shape[1]

print("=== THÔNG TIN BỘ DỮ LIỆU SAU KHI LOAD ===")
print(f"Số lượng mẫu (n): {n_samples} quan trắc")
print(f"Số lượng đặc trưng (p): {p_features} biến")
print(f"Target: CO(GT)")


## 2. Khảo sát dữ liệu (Exploratory Data Analysis - EDA)



In [ ]:
def missing_report(data: pd.DataFrame) -> pd.DataFrame:
    """Báo cáo số lượng và tỷ lệ % missing value
    Args:
        data (pd.DataFrame): Bảng dữ liệu đầu vào cần kiểm tra giá trị thiếu.

    Returns:
        pd.DataFrame: Bảng báo cáo gồm 3 cột ('Đặc trưng', 'Số lượng thiếu', 'Tỷ lệ thiếu (%)')
    """
    missing_count = data.isnull().sum()
    missing_pct = (missing_count / len(data)) * 100
    
    report = pd.DataFrame({
        'Đặc trưng': data.columns,
        'Số lượng thiếu': missing_count,
        'Tỷ lệ thiếu (%)': missing_pct
    }).sort_values(by='Tỷ lệ thiếu (%)', ascending=False)
    report = report.reset_index(drop=True)
    
    report.index = report.index + 1
    
    return report


print("=== 2.1 BẢNG THỐNG KÊ MÔ TẢ TỔNG QUAN ===")
numeric_cols = df.select_dtypes(include=[np.number]).columns
desc_stats = df[numeric_cols].describe().T

desc_stats = desc_stats.rename(columns={
    'count': 'Count', 'mean': 'Mean', 'std': 'Std Dev',
    'min': 'Min', '25%': 'Q1 (25%)', '50%': 'Median (Q2)',
    '75%': 'Q3 (75%)', 'max': 'Max'
})
display(desc_stats.round(2))

print("\n=== 2.2 BÁO CÁO GIÁ TRỊ BỊ THIẾU (MISSING VALUES) ===")
display(missing_report(df))

print("\n=== 2.3 KIỂM TRA DỮ LIỆU TRÙNG LẶP ===")
print(f"Số dòng dữ liệu trùng lặp phát hiện: {df.duplicated().sum()}")



### 2.1. Trực quan hóa phân phối dữ liệu


In [ ]:
# 2.4 BIỂU ĐỒ HISTOGRAM PHÂN PHỐI
print("=== 2.4 BIỂU ĐỒ HISTOGRAM PHÂN PHỐI ===")

df[numeric_cols].hist(bins=30, figsize=(18, 22), color='skyblue', edgecolor='black', layout=(5, 3))

plt.suptitle("Histogram khảo sát phân phối các đặc trưng", fontsize=20, y=1.02)
plt.tight_layout() 
plt.show()

# 2.5 BIỂU ĐỒ BOXPLOT
print("\n=== 2.5 BIỂU ĐỒ BOXPLOT PHÁT HIỆN NGOẠI LAI CHI TIẾT ===")

n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 4))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(data=df, y=col, ax=axes[i], color='lightcoral')
    axes[i].set_title(f'Boxplot: {col}', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('')

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle("Biểu đồ Boxplot khảo sát phân tán và Outliers", fontsize=20, y=1.00)
plt.tight_layout()
plt.show()

### 2.2. Phân tích tương quan tuyến tính


In [ ]:
print("=== 2.6 MA TRẬN TƯƠNG QUAN TUYẾN TÍNH (HEATMAP) ===")
plt.figure(figsize=(12, 10))
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, cbar=True)
plt.title("Ma trận hệ số tương quan Pearson giữa các đặc trưng", fontsize=14)
plt.tight_layout()
plt.show()

print("\n=== 2.7 BIỂU ĐỒ SCATTER PLOT (TOP 5 TƯƠNG QUAN MẠNH NHẤT) ===")
target_col = 'CO(GT)'
top_5_features = corr_matrix[target_col].abs().sort_values(ascending=False).index[1:6]

fig, axes = plt.subplots(1, 5, figsize=(22, 4.5))
for i, feature in enumerate(top_5_features):
    sns.scatterplot(data=df, x=feature, y=target_col, ax=axes[i], color='teal', alpha=0.4)
    axes[i].set_title(f'{target_col} vs {feature}\n(r = {corr_matrix[target_col][feature]:.2f})', fontsize=11)
plt.tight_layout()
plt.show()

### 2.3. Đánh giá ngoại lai (Outliers) bằng phương pháp toán học
1. **Phương pháp IQR:** Dựa trên tứ phân vị (Q1, Q3), phù hợp với dữ liệu bị lệch.
2. **Phương pháp Z-Score:** Lọc các giá trị nằm ngoài mốc $|z| > 3$, giả định dữ liệu có phân phối chuẩn.

In [ ]:
def detect_outliers_iqr(series: pd.Series) -> pd.Index:
    """Phát hiện và trả về chỉ mục (index) của các giá trị ngoại lai dựa trên phương pháp IQR.

    Phương pháp Interquartile Range (IQR) xác định ngoại lai (outliers) là các giá trị 
    nằm ngoài giới hạn an toàn: [Q1 - 1.5 * IQR, Q3 + 1.5 * IQR].
    Trong đó, Q1 và Q3 lần lượt là Tứ phân vị thứ nhất (25%) và thứ ba (75%).

    Args:
        series (pd.Series): Cột dữ liệu (đặc trưng số) cần kiểm tra.

    Returns:
        pd.Index: Tập hợp các chỉ mục (index) của những dòng chứa giá trị ngoại lai.
                  Trường hợp cột rỗng hoặc không có ngoại lai, trả về Index rỗng.
    """
    clean_series = series.dropna()
    if len(clean_series) == 0: return pd.Index([])
    Q1, Q3 = clean_series.quantile(0.25), clean_series.quantile(0.75)
    IQR = Q3 - Q1
    return series[(series < Q1 - 1.5 * IQR) | (series > Q3 + 1.5 * IQR)].index

print("=== 2.8 BÁO CÁO NGOẠI LAI (PHƯƠNG PHÁP IQR) ===")
iqr_outliers = []
for col in numeric_cols:
    idx = detect_outliers_iqr(df[col])
    iqr_outliers.append({
        'Đặc trưng': col,
        'Số lượng Outliers': len(idx),
        'Tỷ lệ (%)': (len(idx) / len(df)) * 100
    })

df_iqr = pd.DataFrame(iqr_outliers).sort_values(by='Tỷ lệ (%)', ascending=False)
df_iqr_display = df_iqr[df_iqr['Số lượng Outliers'] > 0].round(2).reset_index(drop=True)

df_iqr_display.index += 1 
display(df_iqr_display)


print("\n=== 2.9 BÁO CÁO NGOẠI LAI (PHƯƠNG PHÁP Z-SCORE > 3) ===")
z_outliers = []
for col in numeric_cols:
    mean_val, std_val = df[col].mean(), df[col].std()
    if std_val > 0:
        idx = df[abs((df[col] - mean_val) / std_val) > 3].index
        z_outliers.append({
            'Đặc trưng': col,
            'Số lượng Outliers': len(idx),
            'Tỷ lệ (%)': (len(idx) / len(df)) * 100
        })

df_z = pd.DataFrame(z_outliers).sort_values(by='Tỷ lệ (%)', ascending=False)
df_z_display = df_z[df_z['Số lượng Outliers'] > 0].round(2).reset_index(drop=True)

df_z_display.index += 1 
display(df_z_display)